# Kubeflow Notebooks as the Gateway to Kubernetes

### An end-to-end ML workflow: distributed data processing → TPU training → inference

This notebook is the **single control plane** for a realistic ML workflow. From one
JupyterLab Workspace running on a small CPU pod, we drive the full power of a
Kubernetes cluster:

| Stage | What runs | Where | K8s primitive |
|-------|-----------|-------|---------------|
| **1. Data processing** | Apache Spark ETL over Fashion-MNIST | 1 Spark Connect driver + 4 executor pods | Kubeflow SDK `SparkClient` → `SparkConnect` |
| **2. Model training** | Data-parallel JAX training | Multi-host Cloud TPU slice (2 hosts × 4 chips) | Kubeflow Trainer `TrainJob` (JobSet) |
| **3. Inference** | Model server | CPU Deployment (2 replicas) | `Deployment` + `Service` |

All three stages exchange data through one **GCS bucket**. Nothing is mounted:
every stage reads and writes GCS objects with the `google-cloud-storage` Python
SDK, authenticated by **GKE Workload Identity** (the namespace needs
`roles/storage.objectUser` on the bucket — see the next section).

> **The punchline:** the notebook pod itself is *tiny*. It never runs the heavy
> work — it **orchestrates** it. This is Kubeflow Notebooks acting as the
> developer's gateway to the whole cluster.

> Want to pause this workspace mid-workflow without losing kernel state? See
> [`../resumable-notebooks/`](../resumable-notebooks/), which uses a
> WorkspaceKind annotated for GKE Pod Snapshots.


## Architecture

```
                    Kubeflow Notebook (this pod, no accelerator)
                    "the gateway to Kubernetes"
                              │
          ┌───────────────────┼───────────────────────┐
          │ SparkConnect      │ TrainJob               │ Deployment + Service
          ▼                   ▼                        ▼
  ┌───────────────┐   ┌────────────────┐      ┌──────────────────┐
  │ Stage 1: ETL  │   │ Stage 2: Train │      │ Stage 3: Serve   │
  │ Spark:        │   │ 2 TPU hosts    │      │ 2 CPU replicas   │
  │ 1 drv + 4 exec│   │ × 4 chips      │      │ (python:3.12-slim│
  │ (Spark SDK)   │   │ (JobSet)       │      │  + ConfigMap)    │
  └──────┬────────┘   └───────┬────────┘      └────────┬─────────┘
         │ write               │ read / write           │ read
         ▼                     ▼                        ▼
   ┌─────────────────────────────────────────────────────────┐
   │   Shared GCS bucket — google-cloud-storage SDK calls     │
   │   (Workload Identity; no GCSFuse, no PersistentVolume)   │
   │  processed/{train,test}/  model/{params,metrics,ckpt}    │
   └─────────────────────────────────────────────────────────┘
```

Everything below runs from this notebook. Let's go.


## 0. Setup & environment check

We confirm the SDKs are available and that this notebook can talk to the
Kubernetes API using its ServiceAccount.

The next cell locates the directory that holds this notebook's `jobs/` package
(`jobs/pipeline.py`, `jobs/data_processing.py`, `jobs/train.py`, `jobs/serve.py`)
and puts it on `sys.path` so `from jobs import pipeline` works.

> **Upload both together.** The notebook and the `jobs/` folder must sit in the
> *same* directory inside the Workspace (for example `/home/jovyan/distributed/`
> containing `distributed_tpu_example.ipynb` and `jobs/`). If `jobs/` is missing,
> the cell below stops with instructions instead of failing later with a
> confusing `ModuleNotFoundError`.


In [ ]:
import os
import sys
import subprocess

# Ensure required packages (kubeflow[spark], google-cloud-storage) are installed
try:
    from google.cloud import storage  # noqa: F401
    from kubeflow.spark import SparkClient  # noqa: F401
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "kubeflow[spark]", "google-cloud-storage",
    ])


def find_demo_dir():
    """Return the directory that contains the `jobs` package, or None.

    The notebook and the `jobs/` folder are uploaded together into the
    Workspace, but the user can put them in any directory (and the kernel's
    working directory is not necessarily that directory). So we look, in order:
    the current working directory, each of its parents, $HOME, and one level of
    subdirectories under $HOME and under the working directory.
    """
    home = os.path.expanduser("~")
    cwd = os.getcwd()

    candidates = [cwd]
    parent = cwd
    while True:
        nxt = os.path.dirname(parent)
        if nxt == parent:
            break
        parent = nxt
        candidates.append(parent)
    candidates.append(home)
    for base in (cwd, home):
        try:
            candidates.extend(
                os.path.join(base, e) for e in sorted(os.listdir(base))
                if os.path.isdir(os.path.join(base, e))
            )
        except OSError:
            pass

    seen = set()
    for cand in candidates:
        cand = os.path.abspath(cand)
        if cand in seen:
            continue
        seen.add(cand)
        if os.path.isfile(os.path.join(cand, "jobs", "pipeline.py")):
            return cand
    return None


DEMO_DIR = find_demo_dir()
if DEMO_DIR is None:
    raise FileNotFoundError(
        "Could not find the `jobs` package.\n"
        "\n"
        "This notebook needs the `jobs/` folder that ships next to it in\n"
        "examples/distributed/ (jobs/__init__.py, pipeline.py, data_processing.py,\n"
        "train.py, serve.py).\n"
        "\n"
        "Fix: in the JupyterLab file browser, upload the whole `jobs` folder into\n"
        "the SAME directory as this notebook, then re-run this cell. Searched:\n"
        f"  - current working directory: {os.getcwd()}\n"
        f"  - its parent directories\n"
        f"  - {os.path.expanduser('~')} and its immediate subdirectories\n"
        "\n"
        "A quick check from a terminal in the Workspace:\n"
        "  ls -R ~ | head -50   # you should see jobs/pipeline.py"
    )

if DEMO_DIR not in sys.path:
    sys.path.insert(0, DEMO_DIR)

import kubeflow.trainer  # noqa: F401
from jobs import pipeline

print("kubeflow.trainer import OK")
print("DEMO_DIR  =", DEMO_DIR)
# The namespace is read from this pod's mounted ServiceAccount token
# (/var/run/secrets/kubernetes.io/serviceaccount/namespace), so the notebook
# always acts in the namespace the Workspace runs in. Override with DEMO_NAMESPACE.
print("NAMESPACE =", pipeline.NAMESPACE)

# Verify the notebook's ServiceAccount can create the resources we need in its namespace.
for res in [
    "trainjobs.trainer.kubeflow.org",
    "sparkconnects.sparkoperator.k8s.io",
    "deployments.apps",
    "services",
    "configmaps",
]:
    out = subprocess.run(
        ["kubectl", "auth", "can-i", "create", res, "-n", pipeline.NAMESPACE],
        capture_output=True, text=True,
    )
    print(f"can-i create {res.split('.')[0]:17s}:", out.stdout.strip() or out.stderr.strip())


### Provision shared storage (GCS) & grant Workload Identity IAM access

We use a **GCS bucket** as the shared data bus between all three stages (Spark ETL, TPU training, and CPU inference). Every stage talks to it with the `google-cloud-storage` SDK — there is no GCSFuse mount and no shared PersistentVolume.

#### Prerequisite: grant GCS bucket IAM access to your workspace namespace

The notebook always works in **its own namespace**, read at runtime from the pod's ServiceAccount token (`/var/run/secrets/kubernetes.io/serviceaccount/namespace`) — that is the value printed as `NAMESPACE` above. The examples below use `kubeflow-user` as an illustrative name; substitute whatever your Workspace actually printed. (Set `DEMO_NAMESPACE` only if you deliberately want to target a different namespace.)

Before running Stage 1, make sure the bucket exists and that your namespace has `roles/storage.objectUser` on it via **GKE Workload Identity Federation**.

Using `principalSet://.../namespace/${TENANT_NAMESPACE}` grants access to **all** ServiceAccounts in the namespace (covering the workspace pod `ws-<name>`, the Spark driver/executor pods, the TPU `TrainJob` pods, and the inference pods):

```bash
export PROJECT_ID=$(gcloud config get-value project)
export PROJECT_NUMBER=$(gcloud projects describe "${PROJECT_ID}" --format="value(projectNumber)")
export TENANT_NAMESPACE="kubeflow-user"       # <- the namespace printed by the cell above
export GCS_BUCKET="${TENANT_NAMESPACE}-bucket"  # or your custom GCS_BUCKET name

# 1. Create the GCS bucket (if it doesn't exist yet)
gcloud storage buckets create "gs://${GCS_BUCKET}" --location=us-west1 --project="${PROJECT_ID}"

# 2. Grant GCS objectUser access to all pods/ServiceAccounts in the workspace namespace
gcloud storage buckets add-iam-policy-binding "gs://${GCS_BUCKET}" \
  --member="principalSet://iam.googleapis.com/projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${PROJECT_ID}.svc.id.goog/namespace/${TENANT_NAMESPACE}" \
  --role="roles/storage.objectUser"
```

In [ ]:
bucket_name = os.environ.get("GCS_BUCKET", f"{pipeline.NAMESPACE}-bucket")
pipeline.BUCKET_NAME = bucket_name
print(f"Using GCS bucket: gs://{bucket_name}")
print(f"Workspace namespace: {pipeline.NAMESPACE}")
print(
    "\nMake sure you have granted Workload Identity GCS access to this namespace:\n"
    f'  gcloud storage buckets add-iam-policy-binding gs://{bucket_name} \\\n'
    f'    --member="principalSet://iam.googleapis.com/projects/${{PROJECT_NUMBER}}/locations/global/workloadIdentityPools/${{PROJECT_ID}}.svc.id.goog/namespace/{pipeline.NAMESPACE}" \\\n'
    f'    --role="roles/storage.objectUser"'
)

## Stage 1 — Distributed data processing with Apache Spark

We run the ETL as a real **Apache Spark** job on Kubernetes, through the
**Kubeflow Spark Operator**. `SparkClient.connect(...)` opens a **Spark Connect**
session, which makes the operator create a `SparkConnect` custom resource named
`fashion-mnist-etl`; that brings up **1 Spark Connect server ("driver") pod + 4
executor pods**. Spark builds a distributed DataFrame of the 60k Fashion-MNIST
images and normalizes, one-hot encodes, and augments them **in parallel across
the executors**. Each executor then uploads one compressed `.npz` shard directly
to `gs://<bucket>/processed/train/`.

This is the canonical big-data pattern — and from the notebook's perspective
it's just another Kubernetes resource. **This notebook pod stays idle** while
the Spark cluster does the work.

> The driver and executors run the `spark-py312` image from your registry
> (`${REGISTRY}/spark-py312:${TAG}`). If `REGISTRY` is not set in this Workspace,
> the next cell raises a `RuntimeError` telling you how to set it — it will not
> silently try to pull a placeholder image.

Watch it from a terminal while the cell runs:

```bash
kubectl get sparkconnect,pods -n <your-namespace> -w
```


In [ ]:
import importlib
from jobs import pipeline
importlib.reload(pipeline)

# Submit the Spark ETL job (1 driver + 4 executors) and wait for COMPLETED.
data_job = pipeline.run_data_processing(num_executors=4, num_shards=4, wait=True)


In [ ]:
# Inspect the Spark driver logs and the sharded output on the GCS bucket.
pipeline.print_spark_logs()

print("--- shards ---")
from google.cloud import storage
client = storage.Client()
bucket = client.bucket(bucket_name)
blobs = bucket.list_blobs(prefix="processed/train/")
for blob in blobs:
    print(f"gs://{bucket_name}/{blob.name} ({blob.size} bytes)")

print("--- marker ---")
success_blob = bucket.blob("processed/_SUCCESS")
if success_blob.exists():
    print(success_blob.download_as_text())
else:
    print("_SUCCESS marker not found")

## Stage 2 — Distributed model training on TPU

Now the compute-heavy stage. We submit a `TrainJob` that lands on a **multi-host
Cloud TPU v5e slice (2 hosts × 4 chips = 8 TPU chips)**. JAX brings up its
distributed runtime across the hosts, `pmap` parallelizes across the local
chips, and `jax.lax.pmean` averages gradients across *all 8 chips on both hosts*
— true data-parallel training.

The job reads the preprocessed shards from `gs://<bucket>/processed/train/` and
writes the trained model, periodic checkpoints, and metrics back to
`gs://<bucket>/model/` (`params.npz`, `checkpoint-<step>.npz`, `metrics.json`).

> **Before you run this**, two cluster-side things must already exist:
> * the `jax-distributed` ClusterTrainingRuntime — `kubectl get clustertrainingruntimes`
> * the `tpu-v5-8-multi-host` ComputeClass — `kubectl get computeclass tpu-v5-8-multi-host`
>   (apply it with `kubectl apply -f ../compute-classes/tpu-compute-class.yaml`)
>
> **Cost:** this creates a 2-host TPU v5e Spot slice. It is billed for as long as
> the nodes exist. Watch it and clean up when you are done:
>
> ```bash
> kubectl get trainjob -n <your-namespace>
> kubectl get pods -n <your-namespace> -w
> ```
>
> GKE has to auto-create the TPU node pool the first time, which typically takes
> 10+ minutes before the pods leave `Pending`.


In [ ]:
import importlib
from jobs import pipeline
importlib.reload(pipeline)

# Submit the multi-host TPU training job and wait for completion.
train_job = pipeline.run_training(
    num_hosts=2,
    epochs=5,
    global_batch_size=1024,
    wait=True,
)


In [ ]:
# Show training logs and the metrics written to GCS.
pipeline.print_logs(train_job)

import json
from google.cloud import storage
client = storage.Client()
bucket = client.bucket(bucket_name)
blob = bucket.blob("model/metrics.json")
metrics = json.loads(blob.download_as_text())
print("\n=== Final training metrics ===")
print(json.dumps(metrics, indent=2))

## Stage 3 — Inference / serving

The model is trained; now we serve it. We deploy a **2-replica CPU Deployment**
(no TPU needed for this small model) that downloads `model/params.npz` from the
GCS bucket and exposes a `/predict` HTTP endpoint via a `Service`.

The serving code is injected into a stock `python:3.12-slim` image via a
ConfigMap — no custom image build. This shows the same cluster serving traffic
right next to where it trained the model.

The cell below does three things, all from the notebook:

1. builds a ConfigMap named `ml-demo-serve-code` from `jobs/serve.py`,
2. renders `inference-service.yaml` (substituting the bucket name) and applies
   the `Deployment` + `Service`,
3. waits for the rollout.

The equivalent from a terminal, if you prefer:

```bash
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')   # or your namespace
kubectl create configmap ml-demo-serve-code --from-file=serve.py=jobs/serve.py \
  -n "$NS" --dry-run=client -o yaml | kubectl apply -n "$NS" -f -
sed "s/BUCKET_NAME_PLACEHOLDER/<your-bucket>/" inference-service.yaml | kubectl apply -n "$NS" -f -
kubectl rollout status deployment/fashion-mnist-inference -n "$NS" --timeout=180s
```


In [ ]:
# Inject serve.py into the ConfigMap, then apply the Deployment + Service in the workspace namespace.
import os
import subprocess
from jobs import pipeline

# Directory holding this example: it contains jobs/ and inference-service.yaml.
DEMO_DIR = os.path.dirname(os.path.dirname(os.path.abspath(pipeline.__file__)))
serve_path = os.path.join(os.path.dirname(os.path.abspath(pipeline.__file__)), "serve.py")
ns = pipeline.NAMESPACE
bucket_name = pipeline.BUCKET_NAME

# inference-service.yaml sits next to the notebook (examples/distributed/).
# Older layouts kept it in a manifests/ subdirectory, so accept that too.
manifest_candidates = [
    os.path.join(DEMO_DIR, "inference-service.yaml"),
    os.path.join(DEMO_DIR, "manifests", "inference-service.yaml"),
]
manifest_path = next((p for p in manifest_candidates if os.path.isfile(p)), None)
if manifest_path is None:
    raise FileNotFoundError(
        "Could not find inference-service.yaml.\n"
        f"Expected it at: {manifest_candidates[0]}\n"
        f"(also checked: {manifest_candidates[1]})\n"
        "\n"
        "It ships next to this notebook in examples/distributed/. Upload it into\n"
        "the same directory as distributed_tpu_example.ipynb and the jobs/ folder,\n"
        "then re-run this cell."
    )
if not os.path.isfile(serve_path):
    raise FileNotFoundError(
        f"Could not find the serving code at {serve_path}. "
        "Upload the whole jobs/ folder (including serve.py) next to this notebook."
    )
print(f"Using manifest: {manifest_path}")

# Build the ConfigMap from serve.py and apply it (idempotent).
cm = subprocess.run(
    ["kubectl", "create", "configmap", "ml-demo-serve-code",
     f"--from-file=serve.py={serve_path}",
     "-n", ns, "--dry-run=client", "-o", "yaml"],
    check=True, capture_output=True, text=True,
).stdout
subprocess.run(["kubectl", "apply", "-n", ns, "-f", "-"], input=cm, text=True, check=True)

# Read inference-service.yaml and substitute bucket name
with open(manifest_path) as f:
    manifest = f.read()
manifest = manifest.replace("BUCKET_NAME_PLACEHOLDER", bucket_name)

# Apply the Deployment + Service in the workspace namespace.
subprocess.run(["kubectl", "apply", "-n", ns, "-f", "-"], input=manifest, text=True, check=True)
subprocess.run(["kubectl", "rollout", "restart", "deployment/fashion-mnist-inference", "-n", ns], check=False)

subprocess.run(["kubectl", "rollout", "status", "deployment/fashion-mnist-inference", "-n", ns, "--timeout=180s"], check=True)


In [ ]:
# Send real test images to the inference Service and check the predictions.
import os
import json
import numpy as np
import urllib.request
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(bucket_name)
tmp_test = "/tmp/test.npz"
bucket.blob("processed/test/test.npz").download_to_filename(tmp_test)
test = np.load(tmp_test)

sample_idx = [0, 1, 2, 3, 4]
instances = test["images"][sample_idx].tolist()
true_labels = test["labels"][sample_idx].argmax(axis=1).tolist()
os.remove(tmp_test)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

req = urllib.request.Request(
    f"http://fashion-mnist-inference.{pipeline.NAMESPACE}.svc.cluster.local/predict",
    data=json.dumps({"instances": instances}).encode(),
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=30).read())

print("Predicted vs. true:")
for i, (pred, lbl) in enumerate(zip(resp["labels"], true_labels)):
    mark = "✅" if resp["predictions"][i] == lbl else "❌"
    print(f"  {mark} sample {sample_idx[i]}: predicted={pred:12s}  true={CLASS_NAMES[lbl]}")


## Recap — what just happened

From a single small Jupyter Workspace pod, we orchestrated a complete ML
lifecycle across heterogeneous cluster hardware:

- **Stage 1** ran an Apache Spark ETL job (1 Spark Connect driver + 4 executors)
  to preprocess the dataset in parallel — a `SparkConnect` custom resource
  created for us by the Kubeflow Spark SDK.
- **Stage 2** trained a model across 8 Cloud TPU v5e chips on 2 hosts
  (data-parallel training with real cross-host gradient all-reduce).
- **Stage 3** served the model from a 2-replica CPU Deployment.
- All three stages exchanged data through one shared GCS bucket, using the
  `google-cloud-storage` SDK and Workload Identity.

This is the vision: **Kubeflow Notebooks as the gateway to Kubernetes.** The
notebook is where the human works; Kubernetes is the engine that runs the work.

> Want to pause this workspace mid-workflow without losing kernel state? See
> [`../resumable-notebooks/`](../resumable-notebooks/), which uses a
> WorkspaceKind annotated for GKE Pod Snapshots.

### Cleanup

> **Do not skip this.** The TPU slice keeps costing money until the TrainJob and
> its auto-created node pool are gone.

The cell below removes the inference Deployment/Service and the Spark Connect
session, and leaves the data and model in GCS. The full teardown, from a
terminal (`NS` = the `NAMESPACE` printed at the top of this notebook):

```bash
NS=<your-namespace>

# Stage 3
kubectl delete deployment fashion-mnist-inference -n "$NS" --ignore-not-found
kubectl delete service    fashion-mnist-inference -n "$NS" --ignore-not-found
kubectl delete configmap  ml-demo-serve-code      -n "$NS" --ignore-not-found

# Stage 1
kubectl delete sparkconnect fashion-mnist-etl -n "$NS" --ignore-not-found

# Stage 2 (this is the expensive one)
kubectl get trainjob -n "$NS"
kubectl delete trainjob --all -n "$NS"

# Optional: delete the data and model objects
gcloud storage rm -r "gs://<your-bucket>/processed" "gs://<your-bucket>/model"
```

After the TrainJob is deleted, GKE scales the auto-created TPU node pool back
down; confirm with `kubectl get nodes -l cloud.google.com/compute-class=tpu-v5-8-multi-host`.
See the README's Cleanup section for the full list.


In [ ]:
# Optional: tear down the inference service, its ConfigMap, and the SparkConnect session.
# NOTE: this does NOT delete the TrainJob or the TPU nodes — see the commands above.
import subprocess
ns = pipeline.NAMESPACE
subprocess.run(["kubectl", "delete", "deployment", "fashion-mnist-inference", "-n", ns, "--ignore-not-found"], check=False)
subprocess.run(["kubectl", "delete", "service", "fashion-mnist-inference", "-n", ns, "--ignore-not-found"], check=False)
subprocess.run(["kubectl", "delete", "configmap", "ml-demo-serve-code", "-n", ns, "--ignore-not-found"], check=False)
subprocess.run(["kubectl", "delete", "sparkconnect", "fashion-mnist-etl", "-n", ns, "--ignore-not-found"], check=False)

print("\nStill running (delete these to stop TPU billing):")
subprocess.run(["kubectl", "get", "trainjob", "-n", ns], check=False)
